# Исследование надёжности заёмщиков

**Заказчик:** Кредитный отдел банка.

**Цель исследования:** Проверить четыре гипотезы о влиянии социально-демографических факторов на факт погашения кредита в срок:
1.  Существует ли зависимость между количеством детей и возвратом кредита в срок?
2.  Существует ли зависимость между семейным положением и возвратом кредита в срок?
3.  Существует ли зависимость между уровнем дохода и возвратом кредита в срок?
4.  Влияют ли разные цели кредита на его возврат в срок?

**Ход исследования**

Качество входящих данных не гарантировано. Перед проверкой гипотез необходимо провести обзор данных для выявления аномалий, пропусков и артефактов.

Исследование пройдёт в четыре этапа:
1.  **Обзор данных.** Первичный анализ структуры, типов и описательных статистик.
2.  **Предобработка данных.** Обработка пропусков, аномалий, изменение типов, удаление дубликатов и категоризация признаков.
3.  **Исследовательский анализ и проверка гипотез.** Построение сводных таблиц для ответа на ключевые вопросы.
4.  **Общий вывод.**

Результаты исследования будут учтены при построении модели *кредитного скоринга*.

## 1. Обзор данных

Составим первое представление о данных.

In [1]:
import pandas as pd

In [2]:
try:
    df = pd.read_csv('credit_reliability.csv')
    df = df.copy()
    print("Данные успешно загружены")
except FileNotFoundError:
    print("Ошибка: Файл не найден. Проверьте путь к файлу.")

Данные успешно загружены


In [3]:
df.head(10)

,children,days_employed,dob_years,education,education_id,family_status,family_status_id,gender,income_type,debt,total_income,purpose
0,1,-8437.673028,42,высшее,0,женат / замужем,0,F,сотрудник,0,253875.639453,покупка жилья
1,1,-4024.803754,36,среднее,1,женат / замужем,0,F,сотрудник,0,112080.014102,приобретение автомобиля
2,0,-5623.422610,33,Среднее,1,женат / замужем,0,M,сотрудник,0,145885.952297,покупка жилья
3,3,-4124.747207,32,среднее,1,женат / замужем,0,M,сотрудник,0,267628.550329,дополнительное образование
4,0,340266.072047,53,среднее,1,гражданский брак,1,F,пенсионер,0,158616.077870,сыграть свадьбу
5,0,-926.185831,27,высшее,0,гражданский брак,1,M,компаньон,0,255763.565419,покупка жилья
6,0,-2879.202052,43,высшее,0,женат / замужем,0,F,компаньон,0,240525.971920,операции с жильем
7,0,-152.779569,50,СРЕДНЕЕ,1,женат / замужем,0,M,сотрудник,0,135823.934197,образование
8,2,-6929.865299,35,ВЫСШЕЕ,0,гражданский брак,1,F,сотрудник,0,95856.832424,на проведение свадьбы
9,0,-2188.756445,41,среднее,1,женат / замужем,0,M,сотрудник,0,144425.938277,покупка жилья для семьи


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21525 entries, 0 to 21524
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   children          21525 non-null  int64  
 1   days_employed     19351 non-null  float64
 2   dob_years         21525 non-null  int64  
 3   education         21525 non-null  object 
 4   education_id      21525 non-null  int64  
 5   family_status     21525 non-null  object 
 6   family_status_id  21525 non-null  int64  
 7   gender            21525 non-null  object 
 8   income_type       21525 non-null  object 
 9   debt              21525 non-null  int64  
 10  total_income      19351 non-null  float64
 11  purpose           21525 non-null  object 
dtypes: float64(2), int64(5), object(5)
memory usage: 2.0+ MB


Итак, в таблице двенадцать столбцов. Типы данных распределены следующим образом:
- **Целочисленные (int64):** `children`, `dob_years`, `education_id`, `family_status_id`, `debt`.
- **Вещественные (float64):** `days_employed`, `total_income`.
- **Текстовые / Категориальные (object):** `education`, `family_status`, `gender`, `income_type`, `purpose`.

Согласно документации к данным:
- `children` — количество детей в семье;
- `days_employed` — общий трудовой стаж в днях;
- `dob_years` — возраст клиента в годах;
- `education` / `education_id` — уровень образования и его идентификатор;
- `family_status` / `family_status_id` — семейное положение и его идентификатор;
- `gender` — пол клиента;
- `income_type` — тип занятости;
- `debt` — имел ли задолженность по возврату кредитов (целевой признак);
- `total_income` — ежемесячный доход;
- `purpose` — цель получения кредита.

**В названиях колонок нарушений стиля не обнаружено:** все имена записаны в нижнем регистре с использованием знака подчеркивания (snake_case), что соответствует стандарту `PEP 8`.

**Количество значений в столбцах различается.** Это значит, что в данных есть пропущенные значения.
- В столбцах `days_employed` и `total_income` количество ненулевых значений составляет **19 351**.
- Общее количество строк — **21 525**.

**Выявленные аномалии и проблемы (предварительно):**
1.  **Отрицательный стаж:** При беглом осмотре данных ранее были замечены отрицательные значения в столбце `days_employed`. Это артефакт, так как количество отработанных дней не может быть меньше нуля.
2.  **Неявные дубликаты в категориях:** В столбце `education` встречаются значения в разных регистрах (`высшее`, `Среднее`, `ВЫСШЕЕ`). Это требует унификации.
3.  **Тип данных дохода:** Столбец `total_income` имеет тип `float64`. Поскольку доходы обычно измеряются в целых денежных единицах, после обработки пропусков тип следует привести к `int64`.
4.  **Выбросы:** Потенциально могут присутствовать некорректные значения в столбце `children` (например, аномально большое количество детей или нулевой возраст). Это будет проверено на этапе предобработки.

### Выводы по разделу

В каждой строке таблицы содержатся данные об одном клиенте банка: социально-демографические характеристики и факт наличия задолженности (`debt`). Часть колонок описывает самого заёмщика (доход, стаж, образование), часть — параметры кредита (цель).

Предварительно можно утверждать, что данных достаточно для проверки гипотез о влиянии семейного положения, количества детей и дохода на возврат кредита. Однако встречаются пропуски в данных, артефакты в стаже и расхождения в регистрах категориальных признаков.

Чтобы двигаться дальше, необходимо перейти к **предобработке данных**:
1. Заполнить пропуски.
2. Обработать аномалии и артефакты.
3. Изменить типы данных.
4. Удалить дубликаты и привести строки к единому виду.

## 2. Предобработка данных

In [5]:
# Общее количество строк
total_rows = len(df)
print(f"Общее количество записей: {total_rows}")

# Анализ пропусков в days_employed
missing_days = df['days_employed'].isna().sum()
missing_days_pct = (missing_days / total_rows) * 100
print(f"\nСтолбец 'days_employed':")
print(f"  - Пропусков: {missing_days}")
print(f"  - Доля пропусков: {missing_days_pct:.2f}%")

# Анализ пропусков в total_income
missing_income = df['total_income'].isna().sum()
missing_income_pct = (missing_income / total_rows) * 100
print(f"\nСтолбец 'total_income':")
print(f"  - Пропусков: {missing_income}")
print(f"  - Доля пропусков: {missing_income_pct:.2f}%")

# Проверка: совпадают ли строки с пропусками в обоих столбцах
both_missing = df[df['days_employed'].isna() & df['total_income'].isna()].shape[0]
print(f"\nСтрок с пропусками в обоих столбцах одновременно: {both_missing}")

Общее количество записей: 21525

Столбец 'days_employed':
  - Пропусков: 2174
  - Доля пропусков: 10.10%

Столбец 'total_income':
  - Пропусков: 2174
  - Доля пропусков: 10.10%

Строк с пропусками в обоих столбцах одновременно: 2174


Пропуски находятся в **одних и тех же строках** для обоих столбцов. Это важное наблюдение, которое указывает на системный характер отсутствия данных (вероятно, для определённой категории клиентов не заполнялись оба поля одновременно)

Для количественной переменной `total_income` мы заполним пропуски **медианой**. 

*Примечание:* выбор способа заполнения пропусков критически важен, так как он влияет на статистические показатели.

- **Среднее арифметическое (mean):** Сильно чувствительно к **выбросам**. В данных о доходах часто встречаются люди с очень высоким заработком (топ-менеджеры, владельцы бизнеса). Если использовать среднее, то искусственно заполненный доход окажется завышенным и не будет отражать доход «типичного» заёмщика. Это исказит реальную картину.
- **Медиана (median):** Это значение, которое делит выборку пополам (50% значений меньше, 50% больше). Медиана **устойчива к выбросам** (робастна). Даже если в данных есть несколько миллиардеров, медиана останется на уровне дохода обычного человека.

In [6]:
# Заполнение пропусков в total_income медианным значением
median_income = df['total_income'].median()
median_income

145017.93753253992

In [7]:
# Заполняем пропуски
df['total_income'] = df['total_income'].fillna(median_income)

In [8]:
# Проверяем результат
missing_after = df['total_income'].isna().sum()
print(f"Пропусков в total_income после заполнения: {missing_after}")
print(f"Заполнено значений: {missing_income}")

Пропусков в total_income после заполнения: 0
Заполнено значений: 2174


Помимо пропусков в `total_income`, есть пропуски и в `days_employed`. Но там также присутствует аномалия отрицательных значений.

In [9]:
# посмотрим какие вообще значения пренимает столбец
print(f"  Минимальное значение: {df['days_employed'].min():.2f}")
print(f"  Максимальное значение: {df['days_employed'].max():.2f}")

  Минимальное значение: -18388.95
  Максимальное значение: 401755.40


In [10]:
# подсчёт отрицательных значений
negative_days = (df['days_employed'] < 0).sum()
# процент отрицательных значений
negative_pct = (negative_days / len(df)) * 100
print(f"\nОтрицательных значений: {negative_days} ({negative_pct:.2f}%)")


Отрицательных значений: 15906 (73.90%)


In [11]:
# Подсчёт нулевых значений
zero_days = (df['days_employed'] == 0).sum()
zero_pct = (zero_days / len(df)) * 100
print(f"Нулевых значений: {zero_days} ({zero_pct:.2f}%)")

Нулевых значений: 0 (0.00%)


In [12]:
# Подсчёт положительных значений
positive_days = (df['days_employed'] > 0).sum()
positive_pct = (positive_days / len(df)) * 100
print(f"Положительных значений: {positive_days} ({positive_pct:.2f}%)")

Положительных значений: 3445 (16.00%)


In [13]:
# Проверка на аномально большой стаж (больше возраста в днях)
# Максимальный возраст в днях (при возрасте 100 лет)
max_possible_days = 100 * 365
unrealistic_days = (df['days_employed'] > max_possible_days).sum()
print(f"\nЗначений стажа больше 100 лет: {unrealistic_days}")


Значений стажа больше 100 лет: 3445


**Какие аномалии обнаружены?**

При детальном анализе столбца `days_employed` выявлены серьёзные отклонения от реальности:

| Тип значения | Количество | Доля |
| :--- | :--- | :--- |
| Отрицательные значения | 15 906 | 73,90% |
| **Пропуски (NaN)** | **2 174** | **10,10%** |
| **Всего записей** | **21 525** |  |

**Экстремальные значения:**
- Минимальное значение: **-18 388,95 дней**.
- Максимальное значение: **401 755,40 дней** (положительный стаж, эквивалентный примерно **1 100 годам**).
- Количество записей со стажем **более 100 лет** (36 500 дней): **3 445 из 3 445 положительных**, то есть **все положительные значения являются аномально большими**.

**Ключевой вывод:** среди положительных значений нет ни одного реалистичного. Даже минимальный положительный стаж превышает продолжительность человеческой жизни. Это означает, что **весь столбец `days_employed` полностью искажён и не содержит достоверной информации**.

**Возможные причины появления аномалий:**

- **Сбой при выгрузке данных.** При экспорте из внутренней базы данных в CSV-файл мог произойти сдвиг разрядов, добавление лишних символов или неверная интерпретация формата даты.
- **Человеческий фактор.** Операторы колл-центра или сотрудники отделений могли вносить данные с ошибками.

**Почему заполнение медианой в данном случае неприемлемо?**

Стандартный подход к обработке пропусков — заполнение медианным или средним значением. Однако в данной ситуации этот метод приведёт к **грубому искажению данных** по причине, что **доля корректных данных стремится к нулю.** Среди 21 525 записей нет ни одного значения стажа, которое можно было бы признать достоверным. Отрицательные значения — явная ошибка. Положительные значения — аномально велики (все больше 100 лет). Пропуски — отсутствие данных.

Принято решение **полностью исключить столбец `days_employed` из дальнейшего анализа**.

In [14]:
# удаление столбца 
df = df.drop('days_employed', axis=1)

In [15]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21525 entries, 0 to 21524
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   children          21525 non-null  int64  
 1   dob_years         21525 non-null  int64  
 2   education         21525 non-null  object 
 3   education_id      21525 non-null  int64  
 4   family_status     21525 non-null  object 
 5   family_status_id  21525 non-null  int64  
 6   gender            21525 non-null  object 
 7   income_type       21525 non-null  object 
 8   debt              21525 non-null  int64  
 9   total_income      21525 non-null  float64
 10  purpose           21525 non-null  object 
dtypes: float64(1), int64(5), object(5)
memory usage: 1.8+ MB


Заменим вещественный тип данных в столбце `total_income` на целочисленный, так как вещественный тип избыточен для зарплаты.

In [16]:
df['total_income'] = df['total_income'].astype('int64')

In [17]:
# проверяем
df['total_income']

0        253875
1        112080
2        145885
3        267628
4        158616
          ...  
21520    224791
21521    155999
21522     89672
21523    244093
21524     82047
Name: total_income, Length: 21525, dtype: int64

**Проверка дубликатов строк**

In [18]:
# Анализ явных дубликатов
duplicates_count = df.duplicated().sum()
duplicates_pct = (duplicates_count / len(df)) * 100

print(f"Общее количество строк в датафрейме: {len(df)}")
print(f"Количество явных дубликатов (полностью совпадающих строк): {duplicates_count}")
print(f"Доля дубликатов: {duplicates_pct:.2f}%")

Общее количество строк в датафрейме: 21525
Количество явных дубликатов (полностью совпадающих строк): 54
Доля дубликатов: 0.25%


In [19]:
if duplicates_count > 0:
    print("Примеры дублированных строк:")
    display(df[df.duplicated(keep=False)].head(10))

Примеры дублированных строк:


,children,dob_years,education,education_id,family_status,family_status_id,gender,income_type,debt,total_income,purpose
120,0,46,среднее,1,женат / замужем,0,F,сотрудник,0,145017,высшее образование
520,0,35,среднее,1,гражданский брак,1,F,сотрудник,0,145017,сыграть свадьбу
541,0,57,среднее,1,женат / замужем,0,F,сотрудник,0,145017,сделка с подержанным автомобилем
554,0,60,среднее,1,женат / замужем,0,M,сотрудник,0,145017,покупка недвижимости
680,1,30,высшее,0,женат / замужем,0,F,госслужащий,0,145017,покупка жилья для семьи
1005,0,62,среднее,1,женат / замужем,0,F,пенсионер,0,145017,ремонт жилью
1191,0,61,среднее,1,женат / замужем,0,F,пенсионер,0,145017,операции с недвижимостью
1431,0,41,среднее,1,женат / замужем,0,F,сотрудник,0,145017,покупка жилья для семьи
1511,0,58,высшее,0,Не женат / не замужем,4,F,пенсионер,0,145017,дополнительное образование
1681,0,57,среднее,1,гражданский брак,1,F,пенсионер,0,145017,на проведение свадьбы


In [20]:
# удаление явных дубликатов
df = df.drop_duplicates()

In [21]:
# проверка
duplicates_count = df.duplicated().sum()
duplicates_pct = (duplicates_count / len(df)) * 100
print(f"Общее количество строк в датафрейме: {len(df)}")
print(f"Количество явных дубликатов (полностью совпадающих строк): {duplicates_count}")
print(f"Доля дубликатов: {duplicates_pct:.2f}%")

Общее количество строк в датафрейме: 21471
Количество явных дубликатов (полностью совпадающих строк): 0
Доля дубликатов: 0.00%


In [22]:
# Сброс индексов для корректной нумерации
df = df.reset_index(drop=True)

In [23]:
# анализ явных дубликатов
# Список текстовых столбцов для проверки
text_columns = ['education', 'family_status', 'gender', 'income_type', 'purpose']

for col in text_columns:
    print(f"Столбец '{col}':")
    print(f"  Количество уникальных значений: {df[col].nunique()}")
    print(f"  Уникальные значения:")
    for val in sorted(df[col].unique()):
        print(f"    - '{val}'")

Столбец 'education':
  Количество уникальных значений: 15
  Уникальные значения:
    - 'ВЫСШЕЕ'
    - 'Высшее'
    - 'НАЧАЛЬНОЕ'
    - 'НЕОКОНЧЕННОЕ ВЫСШЕЕ'
    - 'Начальное'
    - 'Неоконченное высшее'
    - 'СРЕДНЕЕ'
    - 'Среднее'
    - 'УЧЕНАЯ СТЕПЕНЬ'
    - 'Ученая степень'
    - 'высшее'
    - 'начальное'
    - 'неоконченное высшее'
    - 'среднее'
    - 'ученая степень'
Столбец 'family_status':
  Количество уникальных значений: 5
  Уникальные значения:
    - 'Не женат / не замужем'
    - 'в разводе'
    - 'вдовец / вдова'
    - 'гражданский брак'
    - 'женат / замужем'
Столбец 'gender':
  Количество уникальных значений: 3
  Уникальные значения:
    - 'F'
    - 'M'
    - 'XNA'
Столбец 'income_type':
  Количество уникальных значений: 8
  Уникальные значения:
    - 'безработный'
    - 'в декрете'
    - 'госслужащий'
    - 'компаньон'
    - 'пенсионер'
    - 'предприниматель'
    - 'сотрудник'
    - 'студент'
Столбец 'purpose':
  Количество уникальных значений: 38
  Уникальные зна

In [24]:
# приведение всех текстовых столбцов к нижнему регистру
for col in text_columns:
    df[col] = df[col].str.lower()

In [25]:
print("УНИКАЛЬНЫЕ ЗНАЧЕНИЯ ПОСЛЕ НОРМАЛИЗАЦИИ")

for col in text_columns:
    unique_vals = sorted(df[col].unique())
    print(f"\n{col} ({len(unique_vals)} значений):")
    for val in unique_vals:
        print(f"  - '{val}'")

УНИКАЛЬНЫЕ ЗНАЧЕНИЯ ПОСЛЕ НОРМАЛИЗАЦИИ

education (5 значений):
  - 'высшее'
  - 'начальное'
  - 'неоконченное высшее'
  - 'среднее'
  - 'ученая степень'

family_status (5 значений):
  - 'в разводе'
  - 'вдовец / вдова'
  - 'гражданский брак'
  - 'женат / замужем'
  - 'не женат / не замужем'

gender (3 значений):
  - 'f'
  - 'm'
  - 'xna'

income_type (8 значений):
  - 'безработный'
  - 'в декрете'
  - 'госслужащий'
  - 'компаньон'
  - 'пенсионер'
  - 'предприниматель'
  - 'сотрудник'
  - 'студент'

purpose (38 значений):
  - 'автомобили'
  - 'автомобиль'
  - 'высшее образование'
  - 'дополнительное образование'
  - 'жилье'
  - 'заняться высшим образованием'
  - 'заняться образованием'
  - 'на покупку автомобиля'
  - 'на покупку подержанного автомобиля'
  - 'на покупку своего автомобиля'
  - 'на проведение свадьбы'
  - 'недвижимость'
  - 'образование'
  - 'операции с жильем'
  - 'операции с коммерческой недвижимостью'
  - 'операции с недвижимостью'
  - 'операции со своей недвижимостью'

 Давайте внимательно посмотрим на столбец `purpose` (цель кредита). В нём **38 уникальных значений**, и многие из них — это разные формулировки одной и той же цели.

In [26]:
# фукция для категоризации целей кредита
def categorize_purpose(text):
    if 'автомобил' in text or 'авто' in text:
        return 'операции с автомобилем'
    elif 'жиль' in text or 'недвижим' in text or 'жил' in text or 'ремонт' in text:
        return 'операции с недвижимостью'
    elif 'свадьб' in text:
        return 'проведение свадьбы'
    elif 'образован' in text or 'образовани' in text:
        return 'получение образования'
    else:
        return 'другое'

In [27]:
# применяем фукцию
df['purpose_category'] = df['purpose'].apply(categorize_purpose)

In [28]:
# Проверяем результат
print(f"Количество уникальных категорий: {df['purpose_category'].nunique()}")
print("\nРаспределение по категориям:")
print(df['purpose_category'].value_counts())

Количество уникальных категорий: 4

Распределение по категориям:
purpose_category
операции с недвижимостью    10814
операции с автомобилем       4308
получение образования        4014
проведение свадьбы           2335
Name: count, dtype: int64


In [29]:
# Анализ нестандартного значения 'xna' в столбце gender
# Общее распределение по полу
print("Распределение по полу:")
gender_counts = df['gender'].value_counts()
gender_pct = (gender_counts / len(df)) * 100

for gender in gender_counts.index:
    print(f"  '{gender}': {gender_counts[gender]} записей ({gender_pct[gender]:.2f}%)")

Распределение по полу:
  'f': 14189 записей (66.08%)
  'm': 7281 записей (33.91%)
  'xna': 1 записей (0.00%)


Всего 1 запись с 'xna'. Такое мизерное количество можно смело удалить.

In [30]:
df[df['gender'] == 'xna']

,children,dob_years,education,education_id,family_status,family_status_id,gender,income_type,debt,total_income,purpose,purpose_category
10690,0,24,неоконченное высшее,2,гражданский брак,1,xna,компаньон,0,203905,покупка недвижимости,операции с недвижимостью


In [31]:
# Удаляем строку
df = df[df['gender'] != 'xna']

In [32]:
# укрупнение категорий семейного положения
def simplify_family_status(status):
    if status in ['женат / замужем', 'гражданский брак']:
        return 'в отношениях'
    elif status in ['в разводе', 'вдовец / вдова']:
        return 'был в отношениях'
    elif status == 'не женат / не замужем':
        return 'никогда не состоял'
    else:
        return 'другое'

In [33]:
# Создаём новый столбец с укрупнёнными категориями
df['family_status_group'] = df['family_status'].apply(simplify_family_status)

In [34]:
# Анализ столбца income_type
print("=== АНАЛИЗ СТОЛБЦА income_type ===\n")

# Распределение по типу занятости
income_counts = df['income_type'].value_counts()
income_pct = (income_counts / len(df)) * 100

print("Распределение по типу занятости:")
for income_type, count in income_counts.items():
    print(f"  '{income_type}': {count} записей ({income_pct[income_type]:.1f}%)")

# Предварительный анализ — доля должников по типу занятости
print("\nДоля должников по типу занятости:")
pivot_income = df.pivot_table(index='income_type', values='debt', aggfunc='mean')
pivot_income['debt'] = pivot_income['debt'] * 100
pivot_income.columns = ['Доля должников, %']
print(pivot_income.sort_values('Доля должников, %', ascending=False).round(2))

=== АНАЛИЗ СТОЛБЦА income_type ===

Распределение по типу занятости:
  'сотрудник': 11091 записей (51.7%)
  'компаньон': 5079 записей (23.7%)
  'пенсионер': 3837 записей (17.9%)
  'госслужащий': 1457 записей (6.8%)
  'безработный': 2 записей (0.0%)
  'предприниматель': 2 записей (0.0%)
  'студент': 1 записей (0.0%)
  'в декрете': 1 записей (0.0%)

Доля должников по типу занятости:
                 Доля должников, %
income_type                       
в декрете                   100.00
безработный                  50.00
сотрудник                     9.57
компаньон                     7.40
госслужащий                   5.90
пенсионер                     5.63
предприниматель               0.00
студент                       0.00


In [35]:
# Анализ столбца income_type
print("=== АНАЛИЗ СТОЛБЦА purpose ===\n")

income_counts = df['purpose'].value_counts()
income_pct = (income_counts / len(df)) * 100

for income_type, count in income_counts.items():
    print(f"  '{income_type}': {count} записей ({income_pct[income_type]:.1f}%)")

# Предварительный анализ 
pivot_income = df.pivot_table(index='purpose', values='debt', aggfunc='mean')
pivot_income['debt'] = pivot_income['debt'] * 100
pivot_income.columns = ['Доля должников, %']
print(pivot_income.sort_values('Доля должников, %', ascending=False).round(2))

=== АНАЛИЗ СТОЛБЦА purpose ===

  'свадьба': 793 записей (3.7%)
  'на проведение свадьбы': 773 записей (3.6%)
  'сыграть свадьбу': 769 записей (3.6%)
  'операции с недвижимостью': 675 записей (3.1%)
  'покупка коммерческой недвижимости': 662 записей (3.1%)
  'операции с жильем': 652 записей (3.0%)
  'покупка жилья для сдачи': 652 записей (3.0%)
  'операции с коммерческой недвижимостью': 650 записей (3.0%)
  'жилье': 646 записей (3.0%)
  'покупка жилья': 646 записей (3.0%)
  'покупка жилья для семьи': 638 записей (3.0%)
  'строительство собственной недвижимости': 635 записей (3.0%)
  'недвижимость': 633 записей (2.9%)
  'операции со своей недвижимостью': 627 записей (2.9%)
  'строительство жилой недвижимости': 625 записей (2.9%)
  'покупка своего жилья': 620 записей (2.9%)
  'покупка недвижимости': 620 записей (2.9%)
  'строительство недвижимости': 619 записей (2.9%)
  'ремонт жилью': 607 записей (2.8%)
  'покупка жилой недвижимости': 606 записей (2.8%)
  'на покупку своего автомобиля':

In [36]:
# Анализ столбца income_type
print("=== АНАЛИЗ СТОЛБЦА family_status ===\n")

income_counts = df['family_status'].value_counts()
income_pct = (income_counts / len(df)) * 100

for income_type, count in income_counts.items():
    print(f"  '{income_type}': {count} записей ({income_pct[income_type]:.1f}%)")

# Предварительный анализ 
pivot_income = df.pivot_table(index='family_status', values='debt', aggfunc='mean')
pivot_income['debt'] = pivot_income['debt'] * 100
pivot_income.columns = ['Доля должников, %']
print(pivot_income.sort_values('Доля должников, %', ascending=False).round(2))

=== АНАЛИЗ СТОЛБЦА family_status ===

  'женат / замужем': 12344 записей (57.5%)
  'гражданский брак': 4162 записей (19.4%)
  'не женат / не замужем': 2810 записей (13.1%)
  'в разводе': 1195 записей (5.6%)
  'вдовец / вдова': 959 записей (4.5%)
                       Доля должников, %
family_status                           
не женат / не замужем               9.75
гражданский брак                    9.32
женат / замужем                     7.54
в разводе                           7.11
вдовец / вдова                      6.57


In [37]:
print("УНИКАЛЬНЫЕ ЗНАЧЕНИЯ ПОСЛЕ НОРМАЛИЗАЦИИ")

for col in text_columns:
    unique_vals = sorted(df[col].unique())
    print(f"\n{col} ({len(unique_vals)} значений):")
    for val in unique_vals:
        print(f"  - '{val}'")

УНИКАЛЬНЫЕ ЗНАЧЕНИЯ ПОСЛЕ НОРМАЛИЗАЦИИ

education (5 значений):
  - 'высшее'
  - 'начальное'
  - 'неоконченное высшее'
  - 'среднее'
  - 'ученая степень'

family_status (5 значений):
  - 'в разводе'
  - 'вдовец / вдова'
  - 'гражданский брак'
  - 'женат / замужем'
  - 'не женат / не замужем'

gender (2 значений):
  - 'f'
  - 'm'

income_type (8 значений):
  - 'безработный'
  - 'в декрете'
  - 'госслужащий'
  - 'компаньон'
  - 'пенсионер'
  - 'предприниматель'
  - 'сотрудник'
  - 'студент'

purpose (38 значений):
  - 'автомобили'
  - 'автомобиль'
  - 'высшее образование'
  - 'дополнительное образование'
  - 'жилье'
  - 'заняться высшим образованием'
  - 'заняться образованием'
  - 'на покупку автомобиля'
  - 'на покупку подержанного автомобиля'
  - 'на покупку своего автомобиля'
  - 'на проведение свадьбы'
  - 'недвижимость'
  - 'образование'
  - 'операции с жильем'
  - 'операции с коммерческой недвижимостью'
  - 'операции с недвижимостью'
  - 'операции со своей недвижимостью'
  - 'поку